In [1]:
from dotenv import load_dotenv

load_dotenv()

True

## Summarize messages

In [2]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import SummarizationMiddleware

agent = create_agent(
    model="groq:openai/gpt-oss-20b",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:openai/gpt-oss-20b",
            trigger=("tokens", 100),
            keep=("messages", 1)
        )
    ],
)

In [3]:
from langchain.messages import HumanMessage, AIMessage
from pprint import pprint

response = agent.invoke(
    {"messages": [
        HumanMessage(content="What is the capital of the moon?"),
        AIMessage(content="The capital of the moon is Lunapolis."),
        HumanMessage(content="What is the weather in Lunapolis?"),
        AIMessage(content="Skies are clear, with a high of 120C and a low of -100C."),
        HumanMessage(content="How many cheese miners live in Lunapolis?"),
        AIMessage(content="There are 100,000 cheese miners living in Lunapolis."),
        HumanMessage(content="Do you think the cheese miners' union will strike?"),
        AIMessage(content="Yes, because they are unhappy with the new president."),
        HumanMessage(content="If you were Lunapolis' new president how would you respond to the cheese miners' union?"),
        ]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'messages': [HumanMessage(content='Here is a summary of the conversation to date:\n\n## SESSION INTENT\nThe user is requesting factual information about Lunapolis, the fictional capital on the moon, including its weather, population of cheese miners, and potential union activity.\n\n## SUMMARY\n- The assistant stated that the capital of the moon is **Lunapolis**.  \n- Weather in Lunapolis: **Clear skies**, high of **120\u202f°C** and low of **-100\u202f°C**.  \n- Population of cheese miners in Lunapolis: **100,000**.  \n- The assistant predicted a **union strike** among cheese miners, citing dissatisfaction with the new president.\n\n## ARTIFACTS\nNone.\n\n## NEXT STEPS\nNone.', additional_kwargs={'lc_source': 'summarization'}, response_metadata={}, id='8b40a168-8ed2-4189-b40b-e9dfae4317e3'),
              HumanMessage(content="If you were Lunapolis' new president how would you respond to the cheese miners' union?", additional_kwargs={}, response_metadata={}, id='5e714c89-7a46-470e-af

In [4]:
print(response["messages"][0].content)

Here is a summary of the conversation to date:

## SESSION INTENT
The user is requesting factual information about Lunapolis, the fictional capital on the moon, including its weather, population of cheese miners, and potential union activity.

## SUMMARY
- The assistant stated that the capital of the moon is **Lunapolis**.  
- Weather in Lunapolis: **Clear skies**, high of **120 °C** and low of **-100 °C**.  
- Population of cheese miners in Lunapolis: **100,000**.  
- The assistant predicted a **union strike** among cheese miners, citing dissatisfaction with the new president.

## ARTIFACTS
None.

## NEXT STEPS
None.


## Trim/delete messages

In [5]:
from typing import Any
from langchain.agents import AgentState
from langchain.messages import RemoveMessage
from langgraph.runtime import Runtime
from langchain.agents.middleware import before_agent
from langchain.messages import ToolMessage

@before_agent
def trim_messages(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """Remove all the tool messages from the state"""
    messages = state["messages"]

    tool_messages = [m for m in messages if isinstance(m, ToolMessage)]
    
    return {"messages": [RemoveMessage(id=m.id) for m in tool_messages]}

In [6]:
agent = create_agent(
    model="groq:openai/gpt-oss-20b",
    checkpointer=InMemorySaver(),
    middleware=[trim_messages],
)

In [7]:
response = agent.invoke(
    {"messages": [
        HumanMessage(content="My device won't turn on. What should I do?"),
        ToolMessage(content="blorp-x7 initiating diagnostic ping…", tool_call_id="1"),
        AIMessage(content="Is the device plugged in and turned on?"),
        HumanMessage(content="Yes, it's plugged in and turned on."),
        ToolMessage(content="temp=42C voltage=2.9v … greeble complete.", tool_call_id="2"),
        AIMessage(content="Is the device showing any lights or indicators?"),
        HumanMessage(content="What's the temperature of the device?")
        ]},
    {"configurable": {"thread_id": "2"}}
)

pprint(response)

{'messages': [HumanMessage(content="My device won't turn on. What should I do?", additional_kwargs={}, response_metadata={}, id='e2b97d49-c31a-480e-a278-6d28f9d151b9'),
              AIMessage(content='Is the device plugged in and turned on?', additional_kwargs={}, response_metadata={}, id='106797fd-632d-4133-a0f1-443ba9d5e322', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content="Yes, it's plugged in and turned on.", additional_kwargs={}, response_metadata={}, id='b555ba0f-f855-430d-8121-a71904eb798a'),
              AIMessage(content='Is the device showing any lights or indicators?', additional_kwargs={}, response_metadata={}, id='695f78a7-012c-49f0-bb7e-128c1e83a8cb', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content="What's the temperature of the device?", additional_kwargs={}, response_metadata={}, id='deb850ad-3219-4ab7-8d28-3bd1b85c6eae'),
              AIMessage(content='I’m not able to read the temperature directly. Could you tell 

In [8]:
print(response["messages"][-1].content)

I’m not able to read the temperature directly. Could you tell me which device you’re talking about (e.g., a laptop, smartphone, tablet, or something else) and what you’re seeing on the screen or in the status lights? That’ll help me give you the right steps.
